<!-- Logos Banner -->
<div style="display: flex; align-items: center; justify-content: space-between; padding: 50px 0; border-bottom: 2px solid #003399;">
    <div style="display: flex; align-items: center;">
        <img src="https://raw.githubusercontent.com/geojupyter/jupytergis/main/packages/base/style/icons/logo.svg" alt="JupyterGIS" width="40" style="margin-right: 80px;">
        <img src="https://brand.esa.int/files/2020/05/ESA_logo_2020_Deep-scaled.jpg" alt="ESA" width="150">
    </div>
    <div style="display: flex; align-items: center;">
        <img src="https://quantstack.net/img/quantstack/logo-website.svg" alt="QuantStack" width="150" style="margin-right: 80px;">
        <img src="https://raw.githubusercontent.com/annefou/jupytergis-showcases/refs/heads/main/content/simula-logo.svg" alt="Simula" width="150" style="margin-right: 80px;">
        <img src="https://www.lifewatch.eu/wp-content/uploads/2021/07/logoLW_eric_outline2-01.svg" alt="Lifewatch ERIC" width="150">
    </div>
</div>

<!-- Badges -->
<p align="center">
    <a href="https://usegalaxy.eu/?tool_id=interactive_tool_jupytergis_notebook&version=latest"><img src="https://img.shields.io/badge/launch-Galaxy%20Europe-green?logo=jupyter" alt="Galaxy Europe"></a>
    <a href="https://github.com/geojupyter/jupytergis"><img src="https://img.shields.io/github/stars/geojupyter/jupytergis?style=social" alt="GitHub Stars"></a>
    <a href="https://jupytergis.readthedocs.io/"><img src="https://img.shields.io/badge/docs-readthedocs-blue" alt="Documentation"></a>
    <a href="https://opensource.org/licenses/BSD-3-Clause"><img src="https://img.shields.io/badge/License-BSD%203--Clause-blue.svg" alt="License"></a>
</p>

# RGB Satellite Imagery Styling

**Authors:** Anne Fouilloux

**Affiliation:** QuantStack, Simula Research Laboratory, LifeWatch ERIC  

**Date:** December 2025

**Contract:** ESA 4000144740/24/I-DT-bgh

---

## Purpose

This notebook demonstrates **RGB satellite imagery visualization** with JupyterGIS, showcasing:

- 🛰️ Loading Cloud Optimized GeoTIFF (COG) from European data providers
- 🎨 Adjusting min/max stretch for optimal visualization  
- 🔍 Interactive pan, zoom, and layer controls
- 📊 Multi-band imagery display (RGB + NIR)

> This demo uses **EOxCloudless** - a Sentinel-2 cloudless mosaic produced by [EOX](https://eox.at/), an Austrian Earth Observation company. No data download required - imagery streams directly to your browser.

## Background: Sentinel-2 Cloudless Mosaic

The **EOxCloudless** dataset is a global, cloudless Sentinel-2 mosaic created by:
1. Collecting all Sentinel-2 acquisitions over a time period
2. Automatically detecting and removing clouds
3. Compositing the best pixels into a seamless mosaic


**Technical Details:**

- **Resolution:** 10m (visible bands)
- **Format:** Cloud Optimized GeoTIFF (COG)
- **Bands:** Red, Green, Blue, Near-Infrared (4 bands)
- **Bit depth:** 16-bit unsigned integer
- **Provider:** EOX (Austria) - [eox.at](https://eox.at)



> - [Sentinel-2 on Copernicus](https://sentinels.copernicus.eu/web/sentinel/missions/sentinel-2)
> - [EOxCloudless Product](https://s2maps.eu/)


## Setup

First, we install and import the required libraries.

In [1]:
# Install pystac-client and myst and refresh your browser before running the next cells
!pip install pystac-client jupyterlab-myst -q 

In [2]:
from jupytergis import GISDocument

## Understanding Image Stretch

Satellite sensors capture data in **digital numbers (DN)** that need to be stretched for visualization:

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `min` | Minimum DN mapped to black | 0 - 500 |
| `max` | Maximum DN mapped to white | 1000 - 10000 |
| `normalize` | Scale values to 0-1 range | True/False |

> Why stretch matters
> - **Too narrow** (e.g., max=500): Image appears washed out/saturated
> - **Too wide** (e.g., max=10000): Image appears too dark
> - **Optimal** (e.g., max=1000): Good contrast showing terrain features
>
> The optimal stretch depends on the scene content (water, vegetation, urban, snow, etc.)

## Create the Map

We'll create a JupyterGIS document and add:
1. **OpenStreetMap** basemap for geographic reference
2. **EOxCloudless** Sentinel-2 mosaic with optimized stretch

In [3]:
doc = GISDocument("RGB_Demo.jGIS", latitude=47.0, longitude=11.0, zoom=6)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap"
)

# EOxCloudless with explicit min/max for 16-bit data
doc.add_tiff_layer(
    url="https://s2downloads.eox.at/demo/EOxCloudless/2020/rgbnir/s2cloudless2020-16bits_sinlge-file_z0-4.tif",
    name="EOxCloudless Sentinel-2",
    min=0,
    max=1000,  # 16-bit reflectance values
    opacity=0.9
)

doc

## Interacting with the Map

**Navigation:**
- 🖱️ **Pan:** Click and drag
- 🔍 **Zoom:** Scroll wheel or +/- buttons
- 📍 **Identify:** Press `i` then click on the map

**Layer Controls (Left Panel):**
- 👁️ Toggle layer visibility
- 🎚️ Adjust opacity
- ↕️ Reorder layers by dragging

**Styling (Right-click layer → Symbology):**
- Adjust min/max values
- Change color ramp
- Select different bands for RGB composite


## Exercise: Experiment with Stretch Values

:::{exercise}
:label: stretch-exercise

Try different `max` values to see how they affect the visualization:

1. `max=500` - What happens to bright areas?
2. `max=2000` - How does the image contrast change?
3. `max=5000` - Can you still see terrain details?

Modify the code below and re-run:
:::

:::{solution} stretch-exercise
:class: dropdown
```python
# Try different max values
doc2 = GISDocument("Stretch_Test.jGIS", latitude=47.0, longitude=11.0, zoom=8)

doc2.add_tiff_layer(
    url="https://s2downloads.eox.at/demo/EOxCloudless/2020/rgbnir/s2cloudless2020-16bits_sinlge-file_z0-4.tif",
    name="EOxCloudless (max=2000)",
    min=0,
    max=2000,  # Change this value!
)

doc2
```
:::

## Summary

| Concept | Key Point |
|---------|-----------|
| COG Streaming | Load large satellite imagery without downloading |
| Image Stretch | `min`/`max` control brightness and contrast |
| RGB Display | JupyterGIS composites multi-band imagery automatically |
| Layer Controls | Adjust opacity, visibility, order interactively |

> **📁 Note:** The `.jGIS` file saves automatically. You can reopen it, share with collaborators, or export to QGIS.

---

*This demo was developed as part of ESA Contract 4000144740/24/I-DT-bgh*